# 01: Freeze the existing sample
The workflow's PubMed sampling step is **replaced**: the independent coder must code the same 750 records that Claude Opus 5.5 coded, otherwise no kappa can be computed. This notebook freezes that frame with content hashes and checks it against the Opus code files (same PMIDs, same order). No live query is made.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np
pd.set_option("display.max_colwidth", 80)

In [2]:
import json
from io_utils import load_sample_records, load_opus_codes, source, sha256, FROZEN
rec = load_sample_records()
opus = load_opus_codes(dedupe=False)
assert len(rec) == len(opus) == 750
for arm in "AB":
    assert rec.loc[rec.arm == arm, "pmid"].tolist() == opus.loc[opus.arm == arm, "pmid"].tolist(), arm
# The 2026-09-25 draw contains within-arm duplicates (same PMID drawn twice in Arm B; Opus coded both
# copies 'not relevant'). Each record is coded once here; the duplicates are logged.
dups = rec[rec.duplicated(["arm", "pmid"], keep="first")]
dups.to_csv(FROZEN / "within_arm_duplicates_dropped.tsv", sep="	", index=False)
print("within-arm duplicates dropped:", dups[["arm", "stratum", "pmid"]].values.tolist())
rec = rec.drop(dups.index).reset_index(drop=True)
assert (rec["abstract"].str.strip() != "").all(), "records without abstract"
rec.insert(0, "review_id", [f"{a}{i:03d}" for a, i in zip(rec.arm, rec.groupby("arm").cumcount() + 1)])
rec.to_csv(FROZEN / "sample_frame.tsv", sep="\t", index=False)
hashes = {k: sha256(source(k)) for k in ["sample_json", "opus_codes_A", "opus_codes_B", "kappa_sheet"]}
(FROZEN / "input_hashes.json").write_text(json.dumps(hashes, indent=2))
rec.groupby(["arm", "stratum"]).size()

within-arm duplicates dropped: [['B', '2010-2019', '27959617'], ['B', '2020-2025', '39505152']]


arm  stratum  
A    1990-1994     50
     1995-1999     50
     2000-2004     50
     2005-2009     50
     2010-2014     50
     2015-2019     50
     2020-2025     50
B    1990-1999    100
     2000-2009    100
     2010-2019     99
     2020-2025     99
dtype: int64